# 03_performa — Langkah 12
Jalankan dari atas ke bawah. Sel 1 dan Sel 2 adalah SEL PEMBUKA.

## Sel 1 — SEL PEMBUKA (Python)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%load_ext rpy2.ipython
!apt-get -qq update > /dev/null
!apt-get -qq install -y libpoppler-cpp-dev libglpk-dev libxml2-dev libcurl4-openssl-dev libssl-dev libfontconfig1-dev libharfbuzz-dev libfribidi-dev libfreetype6-dev libpng-dev libtiff5-dev libjpeg-dev > /dev/null
print('Sel 1 selesai')

## Sel 2 — SEL PEMBUKA (R)

In [ ]:
%%R
proj <- "/content/drive/MyDrive/bibliometrik-cxr"
lib  <- file.path(proj, "Rlib")
dir.create(lib, recursive = TRUE, showWarnings = FALSE)
.libPaths(c(lib, .libPaths()))
codename <- system("lsb_release -cs", intern = TRUE)
options(repos = c(CRAN = sprintf("https://packagemanager.posit.co/cran/__linux__/%s/latest", codename)))
options(HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
  paste(getRversion(), R.version$platform, R.version$arch, R.version$os)))
pkgs <- c("bibliometrix", "UpSetR", "readxl", "writexl", "ggplot2", "htmlwidgets")
baru <- pkgs[!pkgs %in% rownames(installed.packages())]
if (length(baru)) install.packages(baru, lib = lib)
setwd(proj)
suppressPackageStartupMessages(library(bibliometrix))
cat("Folder kerja:", getwd(), "\nbibliometrix", as.character(packageVersion("bibliometrix")), "siap\n")

## Sel 3 — Tabel, grafik, dan CAGR

In [ ]:
%%R
library(bibliometrix); library(writexl); library(ggplot2)
M <- readRDS("03_clean/M.rds")

res <- biblioAnalysis(M, sep = ";")
S   <- summary(res, k = 20, pause = FALSE, verbose = FALSE)
lc  <- localCitations(M, sep = ";")
br  <- bradford(M)

write_xlsx(list(
  Info = S$MainInformationDF, Tahunan = S$AnnualProduction,
  Penulis = S$MostProdAuthors, Sitasi = S$MostCitedPapers,
  LocalCited = lc$Papers, Negara = S$MostProdCountries,
  Sumber = S$MostRelSources, Bradford = br$table
), "05_figures/performa.xlsx")

p <- plot(res, k = 10, pause = FALSE)
for (nm in names(p)) {
  ggsave(sprintf("05_figures/performa_%s.png", nm), p[[nm]], width = 8, height = 5, dpi = 300)
}
ggsave("05_figures/performa_bradford.png", br$graph, width = 8, height = 5, dpi = 300)

ann <- table(M$PY)
print(ann)
t0 <- min(as.numeric(names(ann)))
cagr <- (ann[["2025"]] / ann[[as.character(t0)]])^(1 / (2025 - t0)) - 1
cat("CAGR", t0, "-2025:", round(cagr * 100, 1), "%\n")
print(S$MainInformationDF)
list.files("05_figures")